In [ ]:
# Summary statistics and non-parametric tests for roughness and curvature data across anvils and conditions on a balanced dataset where n damaged and undamaged patches are equal per anvil.

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Configuration

root_path   = Path("PATH")
random_seed = 4

group_map = {
    "KBN-AV2":      "KBN_sandstone",
    "KBN-AV3":      "KBN_sandstone",
    "KBN-AV4":      "KBN_sandstone",
    "KBN-AV5":      "KBN_sandstone",
    "KBN-AV6":      "KBN_sandstone",
    "KBY-AV7":      "KBY_sandstone",
    "KBY-AV8":      "KBY_sandstone",
    "KBY-AV9":      "KBY_sandstone",
    "KBY-AV11":     "KBY_sandstone",
    "KBY-AV12":     "KBY_sandstone",
    "AL-AV2":       "AL_limestone_I",
    "AL-AV3":       "AL_limestone_I",
    "AL-AV5_east":  "AL_limestone_I",
    "AL-AV5_west":  "AL_limestone_I",
    "AL-AV8":       "AL_limestone_I",
    "AL-AV7":       "AL_limestone_II",
    "AL-AV9":       "AL_limestone_II",
    "AL-AV12":      "AL_limestone_II",
    "AL-AV13":      "AL_limestone_II",
}

anvil_order = list(group_map.keys())

# Scalar fields to analyse
scalar_fields = {
    "roughness": "Roughness (10)",
    "curvature": "Mean curvature (10)",
}

# Data loading
records = []

for anvil_dir in sorted(root_path.iterdir()):
    if not anvil_dir.is_dir():
        continue
    anvil_id = anvil_dir.name

    output_files = list(anvil_dir.glob("*_output.txt"))
    if not output_files:
        print(f"Skipping {anvil_id} — no output.txt found")
        continue

    output_file = output_files[0]

    with open(output_file, "r") as f: # Assumes the first line with column names starts with "//", corresponding to CC format.
        lines = f.readlines()
    for i, line in enumerate(lines):
        if line.startswith("//"):
            col_names = [c.strip() for c in line.lstrip("/").strip().split(";")]
            df_file = pd.read_csv(output_file, sep=";", skiprows=i + 1, header=None, names=col_names)
            df_file.columns = [c.lstrip("/") for c in df_file.columns]
            break

    missing = [col for col in list(scalar_fields.values()) + ["condition"] if col not in df_file.columns] # Check for required columns
    if missing:
        print(f"Skipping {anvil_id} — missing columns: {missing}")
        continue

    subset = df_file[list(scalar_fields.values()) + ["condition"]].dropna() # Keep only rows with valid data in the relevant columns
    subset["anvil_id"] = anvil_id # Add anvil_id for grouping later
    records.append(subset) # Append to list for concatenation

df = pd.concat(records, ignore_index=True)
if df.empty:
    raise RuntimeError("No data loaded. Check paths and filenames.")

df["RM/locality"] = df["anvil_id"].map(group_map) # Add group labels for later analysis
anvils = [a for a in anvil_order if a in df["anvil_id"].unique()] # Reorder anvils based on predefined order, filtering out any that weren't loaded
print(f"Loaded {len(df):,} rows across {len(anvils)} anvils.")

# Balancing loop: downsample damaged to match undamaged per anvil
balanced_chunks = []

for anvil in anvils:
    sub       = df[df["anvil_id"] == anvil] # Subset for anvil being currently processed
    damaged   = sub[sub["condition"] == "damaged"]
    undamaged = sub[sub["condition"] == "undamaged"]

    if damaged.empty or undamaged.empty:
        print(f"Skipping {anvil} in balancing: missing one condition.")
        continue

    n_balanced    = min(len(damaged), len(undamaged)) # Determine the smaller of the two groups for balanced sampling
    damaged_bal   = damaged.sample(n=n_balanced, random_state=random_seed)
    undamaged_bal = undamaged.sample(n=n_balanced, random_state=random_seed)

    balanced_chunks.append(damaged_bal)
    balanced_chunks.append(undamaged_bal)

    print(f"  {anvil}: n_damaged={len(damaged):,}, n_undamaged={len(undamaged):,} → balanced to {n_balanced:,} each")

df_balanced = pd.concat(balanced_chunks, ignore_index=True) # Combine all balanced chunks into a single DataFrame for analysis
print(f"\nBalanced dataset: {len(df_balanced):,} total rows.")

# Helper functions

def mann_whitney_r(damaged, undamaged): # Returns U statistic, p-value, effect size r, and group sizes for two independent samples using Mann-Whitney U test from scipy.
    u_stat, p_value = mannwhitneyu(damaged, undamaged, alternative="two-sided")
    n1, n2 = len(damaged), len(undamaged)
    z = (u_stat - (n1 * n2) / 2) / np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
    r = abs(z) / np.sqrt(n1 + n2)
    return u_stat, p_value, r, n1, n2

def effect_label(r): # Label effect size r as negligible, small, medium, or large based on Cohen 1988 for non parametric effect size benchmarks.
    if r < 0.1:
        return "negligible"
    elif r < 0.3:
        return "small"
    elif r < 0.5:
        return "medium"
    else:
        return "large"

groups = ["KBN_sandstone", "KBY_sandstone", "AL_limestone_I", "AL_limestone_II"]

def between_group_analysis(df_input, value_col, label):
    group_data = {g: df_input[df_input["RM/locality"] == g][value_col].dropna().values # Dictionary of group labels to their corresponding values for the specified column, filtering out any NaN values.
                  for g in groups}

    stat, p_kw = kruskal(*group_data.values()) # Perform Kruskal-Wallis H-test from scipy package for independent samples across the groups, returning the test statistic and p-value.

    kruskal_row = { # Summary dictionary for the Kruskal-Wallis test results, including the comparison label, H statistic, p-value, and whether the result is significant at the 0.05 level.
        "comparison":      label,
        "H_statistic":     stat,
        "p_value_kruskal": p_kw,
        "significant":     p_kw < 0.05,
    }

    group_list = list(group_data.keys()) # List of group labels for generating pairwise comparisons.

    pairs = [(group_list[i], group_list[j]) # Generate all unique pairs of groups for pairwise Mann-Whitney U tests, ensuring that each pair is only tested once (i < j).
             for i in range(len(group_list)) for j in range(i + 1, len(group_list))]

    pair_rows = [] # List to hold results of pairwise comparisons, which will be converted to a DataFrame later.

    for g1, g2 in pairs:
        u, p, r, n1, n2 = mann_whitney_r(group_data[g1], group_data[g2])
        cles = u / (n1 * n2)
        pair_rows.append({
            "comparison":      label,
            "RM/locality_1":   g1,
            "RM/locality_2":   g2,
            "n_RM/locality_1": n1,
            "n_RM/locality_2": n2,
            "U":               u,
            "p_value":         p,
            "effect_size_r":   r,
            "effect_label":    effect_label(r),
            "CLES":            cles,
        })
    pairwise_df = pd.DataFrame(pair_rows) # Convert the list of pairwise comparison results into a DataFrame for easier manipulation and output.

    if len(pairwise_df) > 1: # Ensure that there are multiple pairwise comparisons before applying FDR correction, as it is not meaningful to correct a single p-value.
        reject_fdr, p_fdr, _, _ = multipletests(pairwise_df["p_value"], method="fdr_bh") # Run FDR on the pairwise p-values.
        pairwise_df["p_corrected_fdr"] = p_fdr # Add the FDR-corrected p-values to the DataFrame.
        pairwise_df["significant_fdr"] = reject_fdr # Add a boolean column indicating whether each pairwise comparison is significant after FDR correction.
    else:
        pairwise_df["p_corrected_fdr"] = pairwise_df["p_value"] # If only one pairwise comparison exists, FDR correction is not applicable, so the original value is used.
        pairwise_df["significant_fdr"] = pairwise_df["p_value"] < 0.05

    return kruskal_row, pairwise_df

def run_analysis(df, field_col, anvils, group_map):
    """Run full analysis pipeline for one scalar field. Returns (summary_df, within_df, kruskal_df, pairwise_df)."""

    # Summary statistics
    summary_records = []

    for anvil in anvils:
        sub = df[df["anvil_id"] == anvil]
        for condition in ["damaged", "undamaged"]:
            cond_sub = sub[sub["condition"] == condition]
            if cond_sub.empty:
                continue
            values = cond_sub[field_col]
            summary_records.append({
                "anvil_id":       anvil,
                "RM/locality":    group_map.get(anvil, ""),
                "condition":      condition,
                "n":              len(values),
                "mean":           values.mean(),
                "median":         values.median(),
                "std":            values.std(),
                "mode_classical": values.mode().iloc[0],
                "min":            values.min(),
                "max":            values.max(),
            })
    summary_df = pd.DataFrame(summary_records)

    # Within-anvil Mann-Whitney
    within_records = []

    for anvil in anvils:
        sub       = df[df["anvil_id"] == anvil]
        damaged   = sub[sub["condition"] == "damaged"][field_col]
        undamaged = sub[sub["condition"] == "undamaged"][field_col]
        if damaged.empty or undamaged.empty:
            print(f"  Skipping {anvil}: missing one condition.")
            continue
        u, p, r, n1, n2 = mann_whitney_r(damaged, undamaged)
        within_records.append({
            "anvil_id":      anvil,
            "RM/locality":   group_map.get(anvil, ""),
            "n_damaged":     n1,
            "n_undamaged":   n2,
            "U":             u,
            "p_value":       p,
            "effect_size_r": r,
            "effect_label":  effect_label(r),
        })
    within_df = pd.DataFrame(within_records)

    reject, p_corrected, _, _ = multipletests(within_df["p_value"], method="bonferroni")
    within_df["p_corrected_bonferroni"] = p_corrected
    within_df["significant_corrected"]  = reject
    within_df["CLES"] = within_df["U"] / (within_df["n_damaged"] * within_df["n_undamaged"])
    within_df_cles = within_df[["anvil_id", "RM/locality", "CLES"]].copy()
    kw_cles, pw_cles = between_group_analysis(within_df_cles, "CLES", "CLES_across_groups")

    # Between-group analyses
    df_damaged   = df[df["condition"] == "damaged"]
    df_undamaged = df[df["condition"] == "undamaged"]
    within_df_r  = within_df[["anvil_id", "RM/locality", "effect_size_r"]].copy()

    kw_damaged,   pw_damaged   = between_group_analysis(df_damaged,   field_col,       "damaged_patches") # Compare using only damaged points.
    kw_undamaged, pw_undamaged = between_group_analysis(df_undamaged, field_col,       "undamaged_patches") # Compare using only undamaged points.
    kw_r,         pw_r         = between_group_analysis(within_df_r,  "effect_size_r", "effect_size_r") # Compare using the within-anvil effect sizes as the input for the Kruskal-Wallis test across groups.

    kruskal_df  = pd.DataFrame([kw_damaged, kw_undamaged, kw_r, kw_cles]) # Combine the Kruskal-Wallis results for damaged, undamaged, and effect size comparisons into a single DataFrame.
    pairwise_df = pd.concat([pw_damaged, pw_undamaged, pw_r, pw_cles], ignore_index=True) # Combine the pairwise comparison results for damaged, undamaged, and effect size comparisons into a single DataFrame.

    return summary_df, within_df, kruskal_df, pairwise_df

# Excel style helpers

header_fill = PatternFill("solid", start_color="D6DCE4", end_color="D6DCE4")
sig_fill    = PatternFill("solid", start_color="E2EFDA", end_color="E2EFDA")
header_font = Font(name="Arial", bold=True, color="000000", size=10)
body_font   = Font(name="Arial", size=10)
center      = Alignment(horizontal="center", vertical="center")
thin_border = Border(
    bottom=Side(style="thin", color="BFBFBF"),
    right=Side(style="thin",  color="BFBFBF"),
)

p_value_cols = {"p_value", "p_corrected_bonferroni", "p_corrected_fdr", "p_value_kruskal"}
sci_format   = "0.000000"

def style_header_row(ws, row, n_cols):
    for col in range(1, n_cols + 1):
        cell = ws.cell(row=row, column=col)
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = center

def style_body_row(ws, row, n_cols, highlight=False):
    for col in range(1, n_cols + 1):
        cell = ws.cell(row=row, column=col)
        cell.font      = body_font
        cell.alignment = center
        cell.border    = thin_border
        if highlight:
            cell.fill = sig_fill

def write_df_to_sheet(ws, df_in, highlight_col=None):
    header_row = ws.max_row
    col_names  = list(df_in.columns)
    ws.append(col_names)
    style_header_row(ws, header_row, len(col_names))

    for _, row_data in df_in.iterrows():
        ws.append(list(row_data))
        data_row  = ws.max_row
        highlight = bool(row_data[highlight_col]) if highlight_col and highlight_col in df_in.columns else False
        style_body_row(ws, data_row, len(col_names), highlight=highlight)

        for col_idx, col_name in enumerate(col_names, start=1):
            if col_name in p_value_cols:
                ws.cell(row=data_row, column=col_idx).number_format = sci_format

    for col_idx, col_name in enumerate(col_names, start=1):
        max_len = max(len(str(col_name)), df_in[col_name].astype(str).str.len().max())
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max_len + 4, 40)

    ws.append([])

# Run analysis and export
wb          = Workbook() # Create a new Excel workbook to store the results of the analyses using openpyxl.
first_sheet = True

for field_name, field_col in scalar_fields.items():
    print(f"\nRunning analysis for: {field_name}")
    summary_df, within_df, kruskal_df, pairwise_df = run_analysis(df_balanced, field_col, anvils, group_map)

    if first_sheet:
        ws = wb.active
        first_sheet = False
    else:
        ws = wb.create_sheet()

    ws.title = f"Summary - {field_name}"
    write_df_to_sheet(ws, summary_df)

    ws = wb.create_sheet(f"Within-Anvil MW - {field_name}")
    write_df_to_sheet(ws, within_df, highlight_col="significant_corrected")

    ws = wb.create_sheet(f"Between-Group KW - {field_name}")
    write_df_to_sheet(ws, kruskal_df, highlight_col="significant")

    ws = wb.create_sheet(f"Between-Group Pairwise - {field_name}")
    write_df_to_sheet(ws, pairwise_df, highlight_col="significant_fdr")

output_path = root_path / "statistical_analysis.xlsx"
df_balanced.to_csv(root_path / "df_balanced.csv", index=False)
wb.save(output_path)
print(f"\nDone. Results saved to:\n  {output_path}")

In [ ]:
# LMM: roughness ~ hardness + (1 | anvil_id)
# Summary and plot of roughness vs. hardness for damaged patches only, with a regression line across all points.

import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

root_path    = Path("PATH")
hardness_path = Path("PATH")  
df_balanced_path   = Path("PATH")

# Load balanced dataset
df_balanced = pd.read_csv(df_balanced_path)

# Load hardness data and compute per-anvil mean
hardness_df = pd.read_excel(hardness_path)
hardness_cols = [c for c in hardness_df.columns if c.startswith("Hardness_value")] # Identify columns containing hardness measurements based on their prefix.
print(f"Hardness columns found: {hardness_cols}")
hardness_df["hardness"] = hardness_df[hardness_cols].median(axis=1) # Compute median hardness across the identified columns for each anvil, resulting in a single hardness value per anvil for use in the mixed model.

# Print a list of median hardness values for each anvil to verify the mapping before merging with the balanced dataframe.
print(hardness_df[["ID", "hardness"]].head())
hardness_map = dict(zip(hardness_df["ID"], hardness_df["hardness"]))

# Add hardness to balanced dataframe
df_balanced["hardness"] = df_balanced["anvil_id"].map(hardness_map)

# Filter to damaged only, rename column for formula compatibility
df_damaged = df_balanced[df_balanced["condition"] == "damaged"].dropna(subset=["hardness", "Roughness (10)"])
df_damaged = df_damaged.rename(columns={"Roughness (10)": "roughness"}) # Rename for formula compatibility in statsmodels, which does not handle parentheses in column names well.

# Fit linear mixed model: roughness ~ hardness + (1 | anvil_id)
model  = smf.mixedlm("roughness ~ hardness", df_damaged, groups=df_damaged["anvil_id"])
result = model.fit()

print(result.summary())

# Compute per-anvil median damaged roughness
df_damaged = df_balanced[df_balanced["condition"] == "damaged"].copy()
df_damaged["hardness"] = df_damaged["anvil_id"].map(hardness_map)

per_anvil = df_damaged.groupby("anvil_id").agg( # Aggregate the damaged subset by anvil_id to compute the median roughness and associated hardness for each anvil, as well as the group label for plotting.
    median_roughness=("Roughness (10)", "median"),
    hardness=("hardness", "first"),
    group=("RM/locality", "first")
).reset_index()

# Plot
fig, ax = plt.subplots(figsize=(10, 8))

colors = {
    "KBN_sandstone":  "#1D9E75",
    "KBY_sandstone":  "#9E781D",
    "AL_limestone_I":  "#1D479E",
    "AL_limestone_II": "#9E1D47",
}

for group, sub in per_anvil.groupby("group"): # Create a scatter plot of median roughness vs. hardness for each anvil, colored by group, with a regression line across all points to visualize the overall trend.
    ax.scatter(sub["hardness"], sub["median_roughness"],
               label=group, color=colors[group], s=60, zorder=3)

# Regression line
m, b = np.polyfit(per_anvil["hardness"], per_anvil["median_roughness"], 1) # Fit a simple linear regression line to the per-anvil median roughness vs. median hardness data.
x_line = np.linspace(per_anvil["hardness"].min(), per_anvil["hardness"].max(), 100) # Generate x values for plotting the regression line, spanning the range of hardness values in the dataset.
ax.plot(x_line, m * x_line + b, color="grey", linestyle="--", linewidth=1) # Add the regression line to the plot, styled as a dashed grey line.

# Formatting
ax.set_xlabel("Hardness (median)")
ax.set_ylim(bottom=0)
ax.set_ylabel("Median roughness [mm]")

# Legend
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
for text in ax.legend_.get_texts():
    if "KBN" in text.get_text():
        text.set_text("Sandstone, Koh Boi Noi")
    elif "KBY" in text.get_text():
        text.set_text("Sandstone, Koh Boi Yai")
    elif "AL_limestone_II" in text.get_text():
        text.set_text("Limestone, type II, Ao Lobi")
    elif "AL_limestone_I" in text.get_text():
        text.set_text("Limestone, type I, Ao Lobi")

plt.tight_layout()
plt.savefig(root_path / "roughness_vs_hardness.png", dpi=300)
plt.show()

In [ ]:
# LMM: curvature ~ hardness + (1 | anvil_id)
# Summary and plot of curvature vs. hardness for damaged patches only, with a regression line across all points.

import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

root_path    = Path("PATH")
hardness_path = Path("PATH")
df_balanced_path   = Path("PATH")

# Load balanced dataset
df_balanced = pd.read_csv(df_balanced_path)

# Load hardness data and compute per-anvil mean
hardness_df = pd.read_excel(hardness_path)
hardness_cols = [c for c in hardness_df.columns if c.startswith("Hardness_value")] # Identify columns containing hardness measurements based on their prefix.
print(f"Hardness columns found: {hardness_cols}")
hardness_df["hardness"] = hardness_df[hardness_cols].median(axis=1) # Compute median hardness across the identified columns for each anvil, resulting in a single hardness value per anvil for use in the mixed model.

# Print a list of median hardness values for each anvil to verify the mapping before merging with the balanced dataframe.
print(hardness_df[["ID", "hardness"]].head())
hardness_map = dict(zip(hardness_df["ID"], hardness_df["hardness"]))

# Add hardness to balanced dataframe
df_balanced["hardness"] = df_balanced["anvil_id"].map(hardness_map)

# Filter to damaged only, rename column for formula compatibility
df_damaged = df_balanced[df_balanced["condition"] == "damaged"].dropna(subset=["hardness", "Mean curvature (10)"])
df_damaged = df_damaged.rename(columns={"Mean curvature (10)": "curvature"}) # Rename for formula compatibility in statsmodels, which does not handle parentheses in column names well.

# Fit linear mixed model: curvature ~ hardness + (1 | anvil_id)
model  = smf.mixedlm("curvature ~ hardness", df_damaged, groups=df_damaged["anvil_id"])
result = model.fit()

print(result.summary())

# Compute per-anvil median damaged curvature
df_damaged = df_balanced[df_balanced["condition"] == "damaged"].copy()
df_damaged["hardness"] = df_damaged["anvil_id"].map(hardness_map)
df_damaged = df_damaged.rename(columns={"Mean curvature (10)": "curvature"})

per_anvil = df_damaged.groupby("anvil_id").agg(
    median_curvature=("curvature", "median"),
    hardness=("hardness", "first"),
    group=("RM/locality", "first")
).reset_index()

per_anvil = df_damaged.groupby("anvil_id").agg( # Aggregate the damaged subset by anvil_id to compute the median curvature and associated hardness for each anvil, as well as the group label for plotting.
    median_curvature=("curvature", "median"),
    hardness=("hardness", "first"),
    group=("RM/locality", "first")
).reset_index()

# Plot
fig, ax = plt.subplots(figsize=(10, 8))

colors = {
    "KBN_sandstone":  "#1D9E75",
    "KBY_sandstone":  "#9E781D",
    "AL_limestone_I":  "#1D479E",
    "AL_limestone_II": "#9E1D47",
}

for group, sub in per_anvil.groupby("group"): # Create a scatter plot of median curvature vs. hardness for each anvil, colored by group, with a regression line across all points to visualize the overall trend.
    ax.scatter(sub["hardness"], sub["median_curvature"],
               label=group, color=colors[group], s=60, zorder=3)

# Regression line
m, b = np.polyfit(per_anvil["hardness"], per_anvil["median_curvature"], 1) # Fit a simple linear regression line to the per-anvil median curvature vs. mean hardness data.
x_line = np.linspace(per_anvil["hardness"].min(), per_anvil["hardness"].max(), 100) # Generate x values for plotting the regression line, spanning the range of hardness values in the dataset.
ax.plot(x_line, m * x_line + b, color="grey", linestyle="--", linewidth=1) # Add the regression line to the plot, styled as a dashed grey line.

# Formatting
ax.set_xlabel("Hardness (median)")
ax.set_ylabel("Median of mean curvature [mm^-1]")
ax.set_ylim(bottom=0)

# Limi the number fo decimal points on the y axis label to 3
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.3f}"))

# Legend
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
for text in ax.legend_.get_texts():
    if "KBN" in text.get_text():
        text.set_text("Sandstone, Koh Boi Noi")
    elif "KBY" in text.get_text():
        text.set_text("Sandstone, Koh Boi Yai")
    elif "AL_limestone_II" in text.get_text():
        text.set_text("Limestone, type II, Ao Lobi")
    elif "AL_limestone_I" in text.get_text():
        text.set_text("Limestone, type I, Ao Lobi")
        
plt.tight_layout()
plt.savefig(root_path / "curvature_vs_hardness.png", dpi=300)
plt.show()

In [ ]:
# LMM with interaction: roughness ~ hardness * condition (damaged/undamaged) + (1 | anvil_id)
# LMM with interaction: curvature ~ hardness * condition (damaged/undamaged) + (1 | anvil_id)
# Summary and plot of roughness and curvature vs. hardness for damaged and undamaged patches, with separate regression lines per condition.
# The regression lines do not cross, there is no interaction. Therefore, the interaction term should be dropped from the model.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from matplotlib.lines import Line2D
from pathlib import Path
from matplotlib.legend_handler import HandlerTuple

root_path     = Path("PATH")
hardness_path = Path("PATH")
df_balanced_path   = Path("PATH")

# Load balanced dataset
df_balanced = pd.read_csv(df_balanced_path)

# Load hardness
hardness_df   = pd.read_excel(hardness_path)
hardness_cols = [c for c in hardness_df.columns if c.startswith("Hardness_value")]
hardness_df["hardness"] = hardness_df[hardness_cols].median(axis=1)
hardness_map  = dict(zip(hardness_df["ID"], hardness_df["hardness"]))

# Add hardness and rename columns
df_balanced["hardness"] = df_balanced["anvil_id"].map(hardness_map)
df_model = df_balanced.rename(columns={
    "Roughness (10)":      "roughness",
    "Mean curvature (10)": "curvature",
})
df_model = df_model.dropna(subset=["hardness", "roughness", "curvature"])
df_model["condition"] = df_model["condition"].astype("category")

# Per-anvil summaries for plotting
per_anvil = df_model.groupby(["anvil_id", "condition"]).agg(
    median_roughness=("roughness", "median"),
    median_curvature=("curvature", "median"),
    hardness=("hardness", "first"),
    group=("RM/locality", "first")
).reset_index().dropna(subset=["hardness"])

# Fit models
print("Roughness")
model_r  = smf.mixedlm("roughness ~ hardness * condition", df_model, groups=df_model["anvil_id"])
result_r = model_r.fit()
print(result_r.summary())

print("\nCurvature")
model_c  = smf.mixedlm("curvature ~ hardness * condition", df_model, groups=df_model["anvil_id"])
result_c = model_c.fit()
print(result_c.summary())

# Styling
group_colors = {
    "KBN_sandstone":   "#1D9E75",
    "KBY_sandstone":   "#9E781D",
    "AL_limestone_I":  "#1D479E",
    "AL_limestone_II": "#9E1D47",
}
group_label_map = {
    "AL_limestone_I":  "Limestone, type I, Ao Lobi",
    "AL_limestone_II": "Limestone, type II, Ao Lobi",
    "KBN_sandstone":   "Sandstone, Koh Boi Noi",
    "KBY_sandstone":   "Sandstone, Koh Boi Yai",
}
condition_linestyles = {"damaged": "-", "undamaged": "--"}
condition_markers    = {"damaged": "o", "undamaged": "s"}

x_range = np.linspace(per_anvil["hardness"].min(), per_anvil["hardness"].max(), 100)

def plot_interaction(ax, result, per_anvil, y_col, ylabel, format_decimals=None):
    params = result.params

    # Scatter points: colour by group, shape by condition
    plotted_groups = set()
    for (group, condition), sub in per_anvil.groupby(["group", "condition"]):
        ax.scatter(sub["hardness"], sub[y_col],
                   color=group_colors[group],
                   marker=condition_markers[condition],
                   s=60, zorder=3,
                   label=group_label_map[group] if group not in plotted_groups else "_nolegend_")
        plotted_groups.add(group)

    # Regression lines per condition
    for condition in ["damaged", "undamaged"]:
        intercept = params["Intercept"]
        b_hard    = params["hardness"]
        if condition == "undamaged":
            cond_offset = params.get("condition[T.undamaged]", 0)
            interaction = params.get("hardness:condition[T.undamaged]", 0)
            y_line = (intercept + cond_offset) + (b_hard + interaction) * x_range
        else:
            y_line = intercept + b_hard * x_range
        ax.plot(x_range, y_line,
                color="grey",
                linestyle=condition_linestyles[condition],
                linewidth=1.5)

    # Build legend: coloured circles for groups + grey markers for condition at the bottom
    handles, labels = ax.get_legend_handles_labels()

    condition_handle = (
        Line2D([0], [0], marker="o", color="grey", linestyle="None", markersize=6),
        Line2D([0], [0], marker="s", color="grey", linestyle="None", markersize=6),
    )
    handles.append(condition_handle)
    labels.append("Damaged / Undamaged")

    ax.legend(handles, labels,
              handler_map={tuple: HandlerTuple(ndivide=None)},
              loc="center left", bbox_to_anchor=(1, 0.5),
              fontsize=8, title_fontsize=8)

    ax.set_xlabel("Hardness (median)")
    ax.set_ylabel(ylabel)
    ax.set_ylim(bottom=0)

    if format_decimals is not None:
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.{format_decimals}f}"))

# Plot — figsize=(20, 8) gives each subplot approximately 10x8
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

plot_interaction(axes[0], result_r, per_anvil, "median_roughness", "Median roughness [mm]")

plot_interaction(axes[1], result_c, per_anvil, "median_curvature", "Median of mean curvature [mm^-1]",
                 format_decimals=3)

plt.tight_layout()
plt.savefig(root_path / "interaction_hardness_condition.png", dpi=300)
plt.show()